# Practical 05 — Business Valuation (Monte-Carlo DCF + Comparables, run offline)

In the agentic version of this practical you type `/valuation AAPL` and a fleet of
Claude Code agents values a public company through three lanes — a Monte-Carlo DCF,
a dual-source comparables lane (LLM-proposed peers + 10-K-embedding-similarity
peers), and a qualitative/risk lane — then reconciles them into a fair-value
**median + P10–P90** and benchmarks it against the real market price. All the
arithmetic lives in `tools/*.py`; the LLM only chooses inputs and reads outputs.

Colab can't run the agentic loop, so this notebook drives those same reference
tools **directly and fully offline**: we use the bundled `companyfacts_TEST.json`
fixture instead of SEC EDGAR, inject a tiny deterministic numpy `embed` function
instead of downloading `sentence-transformers`, and inject a fake price fetcher
instead of calling yfinance. You will see the whole EDGAR→financials→comps/
embeddings→Monte-Carlo DCF→reconcile pipeline run on real numbers.

In [ ]:
# Colab: run once to install this practical's dependencies.
# (Locally, skip if you ran `pip install -r requirements.txt`.)
%pip install numpy

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "05-business-valuation"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

In [ ]:
# Common imports plus every tool module. Each tool is also a CLI (`python tools/x.py`),
# but here we call its library functions directly so nothing touches the network.
import json
from pathlib import Path
import numpy as np

import financials              # normalize raw XBRL company facts -> clean financials
import embeddings             # build_index / nearest for 10-K similarity peers
import comps                  # peer_metrics + implied_values (comparables lane)
import montecarlo_dcf as mc   # run_dcf (Monte-Carlo discounted cash flow lane)
import market_price           # get_price (accepts an injectable fetcher -> offline)
import reconcile              # pool the lanes into one fair-value distribution

def show(obj):
    """Print a result dict compactly, dropping bulky per-sample arrays."""
    print(json.dumps({k: v for k, v in obj.items() if k != "samples"}, indent=2))

## Step 1 — EDGAR fetch (offline via the bundled fixture)

Normally `edgar_fetch.py` downloads `companyfacts/CIK##########.json` and the
latest 10-K text from SEC EDGAR (requiring a `SEC_USER_AGENT`). Offline we load
the bundled `tests/fixtures/companyfacts_TEST.json` — the same fixture the test
suite uses. It is a trimmed XBRL "company facts" document: a tree of
`facts -> us-gaap -> <tag> -> units -> USD/shares -> [annual rows]`.

In [ ]:
FIX = root / "tests" / "fixtures" / "companyfacts_TEST.json"
facts = json.loads(FIX.read_text())
print("entityName :", facts["entityName"])
print("cik        :", facts["cik"])
print("us-gaap tags:")
for tag in facts["facts"]["us-gaap"]:
    print("   -", tag)

## Step 2 — Normalize financials

`financials.normalize(facts, ticker=...)` walks the tag-priority lists in
`financials.TAGS`, keeps only `10-K` rows, takes the latest fiscal year, and
derives the inputs every valuation lane needs: revenue, EBIT, D&A, capex,
change in net working capital, debt, cash, shares, and an effective tax rate
(clamped to [0, 0.45]). This is the single source of truth for the company.

In [ ]:
fin = financials.normalize(facts, ticker="TEST")
show(fin)

## Step 3 — Comparables lane, source A: 10-K embedding similarity

The real comps-analyst finds peers two ways. Source A embeds each company's 10-K
narrative and picks the nearest neighbours by cosine similarity. The production
code uses `sentence-transformers`; `embeddings.build_index` and `embeddings.nearest`
both accept an `embed` callable, so we inject a small deterministic bag-of-words
hashing embedder — no model download, identical API.

In [ ]:
_STOP = {"we", "and", "at", "for", "to", "with", "the", "a", "of", "our", "by", "in"}

def embed(texts, dim=64):
    """Deterministic L2-normalized hashed bag-of-words embedding (offline stand-in).
    Drops glue words so similarity tracks shared *content* vocabulary."""
    import hashlib
    out = np.zeros((len(texts), dim))
    for i, t in enumerate(texts):
        for w in str(t).lower().split():
            if w in _STOP:
                continue
            h = int(hashlib.md5(w.encode()).hexdigest(), 16) % dim
            out[i, h] += 1.0
        n = np.linalg.norm(out[i])
        if n > 0:
            out[i] /= n
    return out

# Tiny offline "universe" of 10-K-style narratives (no EDGAR download needed).
universe_items = [
    {"ticker": "DEVCO", "cik": 111, "text":
        "we design and sell consumer hardware devices and accessories at high gross margin"},
    {"ticker": "CHIPCO", "cik": 222, "text":
        "we design semiconductor chips and processors for hardware devices and accessories"},
    {"ticker": "RETAILCO", "cik": 333, "text":
        "we operate retail grocery stores selling consumer food and household staples"},
    {"ticker": "BANKCO", "cik": 444, "text":
        "we provide banking lending and payment services to businesses"},
]
target_text = ("we design and sell premium consumer hardware devices accessories "
               "at high gross margin")

vecs, items = embeddings.build_index(universe_items, embed=embed)
target_vec = embed([target_text])[0]
embed_peers = embeddings.nearest(target_vec, vecs, items, k=4)
print("embedding-similarity peers (nearest 10-K narratives, ranked by cosine):")
print(json.dumps(embed_peers, indent=2))

Interpret: the device company `TEST` lands closest to `DEVCO` and `CHIPCO`
(shared hardware / devices / accessories vocabulary), `RETAILCO` is mid (only
"consumer" overlaps), and `BANKCO` ranks last (no shared content words). That is
exactly the "do LLM-proposed peers agree with embedding peers?" question from the
README — here the embeddings cleanly cluster the hardware names at the top.

## Step 4 — Comparables lane, source B: implied value from peer multiples

Given peers, `comps.peer_metrics(fin, price)` turns each peer's financials + price
into `EV/EBITDA` and `P/E`. `comps.implied_values(target, multiples, ...)` then
Monte-Carlos those multiples onto the target's EBITDA and net income to get a
per-share distribution. We have no offline peer prices, so we build a couple of
synthetic peers (the test suite does the same trick) with hand-set prices.

In [ ]:
# Two synthetic peers: slight variations on the target, each with a quoted price.
peerA = dict(fin, shares=120.0, total_debt=350.0, cash=180.0,
             ebitda=420.0, net_income=260.0)
peerB = dict(fin, shares=80.0, total_debt=500.0, cash=150.0,
             ebitda=360.0, net_income=210.0)
multiples = [comps.peer_metrics(peerA, price=11.0),
             comps.peer_metrics(peerB, price=14.0)]
print("peer multiples:")
print(json.dumps(multiples, indent=2))

comps_res = comps.implied_values(fin, multiples, seed=7, n=4000)
comps_res["source"] = "comps"
print("\ncomparables-implied per-share value:")
show(comps_res)

## Step 5 — Monte-Carlo DCF lane

`montecarlo_dcf.run_dcf(fin, cfg, seed, n)` draws `n` paths for revenue growth,
operating margin, WACC, and terminal growth from the priors in `cfg`, projects
unlevered free cash flow for `cfg["years"]` years plus a Gordon terminal value,
discounts to enterprise value, bridges to equity (− debt + cash), and divides by
shares. It returns the per-share median and P10/P90. (It guards WACC > terminal
growth so the terminal value can't blow up.)

In [ ]:
cfg = {
    "years": 5,
    "revenue_growth":   {"dist": "normal", "mean": 0.05,  "sd": 0.02},
    "operating_margin": {"dist": "normal", "mean": 0.27,  "sd": 0.03},
    "wacc":             {"dist": "normal", "mean": 0.09,  "sd": 0.01},
    "terminal_growth":  {"dist": "normal", "mean": 0.025, "sd": 0.004},
    "tax_rate": fin["tax_rate"],
}
dcf = mc.run_dcf(fin, cfg, seed=7, n=4000)
print("Monte-Carlo DCF per-share value:")
show(dcf)

### Try-it: edit a DCF prior and watch the range move

A README suggestion: nudge a prior and see the distribution shift. Raising the
WACC mean from 9% to 11% discounts future cash flows harder, so every percentile
of the fair-value range should drop.

In [ ]:
cfg_high_wacc = dict(cfg, wacc={"dist": "normal", "mean": 0.11, "sd": 0.01})
dcf_high = mc.run_dcf(fin, cfg_high_wacc, seed=7, n=4000)
print(f"WACC mean 9%  -> median {dcf['median']:.2f}  "
      f"[P10 {dcf['p10']:.2f}, P90 {dcf['p90']:.2f}]")
print(f"WACC mean 11% -> median {dcf_high['median']:.2f}  "
      f"[P10 {dcf_high['p10']:.2f}, P90 {dcf_high['p90']:.2f}]")
print("higher WACC lowered the median:", dcf_high["median"] < dcf["median"])

## Step 6 — Reconcile the lanes

`reconcile.pool(dcf, comps_list, weights, seed, n)` doesn't average point
estimates — it resamples each lane's raw `samples` array in proportion to its
weight and pools them, so the final P10–P90 honestly reflects both lanes'
uncertainty. We weight the DCF 60% and the comparables 40%.

In [ ]:
final = reconcile.pool(dcf, [comps_res],
                       weights={"dcf": 0.6, "comps": 0.4}, seed=7, n=4000)
print("reconciled fair value:")
print(json.dumps(final, indent=2))

## Step 7 — Benchmark against the market price

`market_price.get_price(ticker, fetcher=...)` normally calls yfinance, but it
accepts an injectable `fetcher`, so offline we hand it a fixed quote. We then
compute the gap between our reconciled fair value and the "market" price: a
positive gap means our model sees the stock as undervalued.

In [ ]:
def fake_fetcher(ticker):
    return {"ticker": ticker.upper(), "price": 9.50,
            "market_cap": 9.50 * fin["shares"], "currency": "USD"}

quote = market_price.get_price("TEST", fetcher=fake_fetcher)
fair = final["median"]
price = quote["price"]
gap = (fair - price) / price
print(json.dumps(quote, indent=2))
print(f"\nfair value (median): {fair:.2f}")
print(f"market price        : {price:.2f}")
print(f"gap                 : {gap:+.1%} "
      f"({'undervalued' if gap > 0 else 'overvalued'} per the model)")
verdict = "inside" if final["p10"] <= price <= final["p90"] else "outside"
print(f"market price is {verdict} the model's P10–P90 band "
      f"[{final['p10']:.2f}, {final['p90']:.2f}]")

## Recap / next steps

We ran the full reference pipeline offline, directly on the bundled fixture:

1. **EDGAR fetch** — loaded `companyfacts_TEST.json` (stands in for the EDGAR download).
2. **Financials** — `financials.normalize` turned raw XBRL into clean valuation inputs.
3. **Comps / embeddings** — `embeddings.build_index`/`nearest` (with an injected
   deterministic `embed`) found similar 10-K narratives; `comps.peer_metrics` +
   `comps.implied_values` produced a multiples-based per-share distribution.
4. **Monte-Carlo DCF** — `montecarlo_dcf.run_dcf` simulated a per-share fair-value range.
5. **Reconcile** — `reconcile.pool` blended the lanes into one median + P10–P90.
6. **Benchmark** — `market_price.get_price` (injected fetcher) gave the market gap.

Next steps to explore (offline): change the lane weights in `reconcile.pool`;
add a third synthetic peer or a loss-making target (the comps lane skips P/E when
net income ≤ 0); widen the DCF priors' `sd` to see the band fan out; or swap in a
real company by setting `SEC_USER_AGENT` and calling the `edgar_fetch`/`market_price`
CLIs to replace the fixtures with live data.
print("Pipeline complete — all lanes ran offline.")